# Repository as Policy with RHO

CaP-X generates a new robot program at test time. RHO instead evaluates and evolves a repository before deployment, so the deployed policy no longer needs an LLM call.

This is a **micro-RHO demonstration**, not a reproduction of the paper's 200-generation run. One bounded HELIX generation asks OpenCode and the local `Gemma-4-E2B-it-GGUF` model to mutate a two-file CaP-X policy, gates the child on a fixed training layout, and checks it on a held-out layout. The workshop pod's process timeout and non-root boundary are weaker isolation than the paper's REaaS/evaluator-sidecar architecture.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/ryzers")
import rho_demo

servers = rho_demo.ensure_services()

In [ ]:
ROOT = rho_demo.prepare_workshop()

for name in ("solver/geometry.py", "solver/policy.py", "opencode.json", "helix.toml"):
    print(f"\n--- {name} ---")
    print((ROOT / name).read_text())

In [ ]:
seed_train = rho_demo.score_candidate(ROOT, "train", capture=True)
seed_train

In [ ]:
live_run = rho_demo.run_helix(ROOT, generations=1, timeout_seconds=480)
print(f"exit={live_run.returncode}  timed_out={live_run.timed_out}  elapsed={live_run.elapsed_seconds:.1f}s")

In [ ]:
from IPython.display import Video, display

summary = rho_demo.summarize_run(ROOT)
if summary["improved_best"]:
    comparison_root = Path(summary["live_best"])
    outcome = "LIVE CHILD ACCEPTED AND BECAME THE BEST CANDIDATE"
    mutation = summary["semantic_mutation"]
    shown_diff = summary["best_diff"]
elif summary["accepted"] and not live_run.timed_out:
    comparison_root = Path(summary["live_best"])
    outcome = "LIVE CHILD ACCEPTED TO THE FRONTIER; THE SEED REMAINED BEST"
    mutation = summary["semantic_mutation"]
    shown_diff = summary["child_diff"]
else:
    comparison_root = Path(summary["fallback"]["candidate"])
    outcome = "LIVE CHILD REJECTED OR UNAVAILABLE"
    print(summary["fallback"]["label"])
    mutation = summary["fallback"]["trace"]["mutation"]
    shown_diff = rho_demo.source_diff(ROOT, comparison_root)

print(outcome)
print("Semantic mutation:", *mutation, sep="\n  - ")
print("\nSource diff:\n", shown_diff or "(no source change)")

seed_heldout = rho_demo.score_candidate(ROOT, "val", capture=True)
if comparison_root.resolve() == ROOT.resolve():
    best_heldout = seed_heldout
else:
    best_heldout = rho_demo.score_candidate(comparison_root, "val", capture=True)
print(f"held-out reward: seed={seed_heldout['reward']}  comparison={best_heldout['reward']}")

for label, result in (("seed", seed_heldout), ("comparison", best_heldout)):
    print(label, result["feedback"])
    if result.get("video"):
        display(Video(result["video"], embed=True, width=480))

In [ ]:
# Optional: set True to propose one more child from the saved frontier.
# One or two mutations show repository evolution and gating, not the paper's
# evolutionary performance, which required tens to hundreds of generations.
RUN_SECOND_GENERATION = False

if RUN_SECOND_GENERATION:
    second_run = rho_demo.run_helix(ROOT, generations=2, timeout_seconds=480)
    print(rho_demo.summarize_run(ROOT))
else:
    print("Second generation skipped (workshop default).")